# Experiment 7f — Pure-DL submission (gated MLP from 7e)

Production refit + leaderboard submission for the L1-gated MLP that landed at CV-WAPE **14.593** in `kp-experiment-7e-mlp-gates.ipynb` (§6). This is the pure-DL submission for the grade — **no ensemble with LGBM**. The 7c ensemble (LGBM+MLP at LB 14.68) stays as a separate, already-shipped artifact; this experiment delivers what the professor specifically asked for: a deep-learning model that figures out feature selection on its own.

Two things frozen from 7e (changing either would invalidate the diagnostic chain):

- **Architecture** — identical `GatedSeriesMLP` with the same `gate_idx_map` (11 learnable logits over the 11 domain covariates, applied to every feature derived from each covariate).
- **`GATE_L1_LAMBDA = 0.01`** — the regularization strength that produced the clean gate ordering in 7e §7. Higher would over-suppress, lower would muddy the feature-importance signal.

Production-only changes vs 7e:

- **5 seeds** (production convention, matching exp-5.1 and 7c) instead of 3.
- **Refit on the full train** (no CV holdout), then predict on `validation_input.csv`.
- **Combined-frame feature engineering** — same trick as 7c so lag/lead/rolling features at the start of the val window have proper context from the tail of train.

Outputs:

- `validation_predictions_7f.csv` — standard `(series_id, timestamp, prediction)` submission file.
- `val_components_7f.csv` — per-seed predictions + per-seed learned gate values, for any downstream diagnostic work.

Expected leaderboard: CV→LB gap in this project has been 0.1–0.2. CV here is 14.59, so a leaderboard around **14.7–14.8** is on-trend. The 7c ensemble at LB 14.68 is the ceiling we're competing against; matching or beating it solo would be excellent.

---
## §0. Configuration

In [1]:
from pathlib import Path

DATA_DIR     = Path("/kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data")
OUTPUT_DIR   = Path("/kaggle/working")
# DATA_DIR   = Path("data")
# OUTPUT_DIR = Path("experiment-7")

TRAIN_PATH     = DATA_DIR / "train.csv"
VAL_INPUT_PATH = DATA_DIR / "validation_input.csv"
VAL_INDEX_PATH = DATA_DIR / "forecast_index_validation.csv"

SEED         = 42
SEEDS        = [42, 101, 202, 303, 404]   # production 5-seed ensemble
EVAL_HORIZON = 336

DOMAIN_FEAT_COLS = [
    "workload_intensity", "demand_forecast", "upstream_quality_forecast",
    "promotion_intensity", "shock_risk", "maintenance_known",
    "unit_reliability_forecast", "queue_pressure_forecast",
    "network_pressure_forecast", "event_load_forecast",
    "service_irregularity_risk_forecast",
]
TIME_COLS = ["hour_sin", "hour_cos", "dow_sin", "dow_cos",
             "doy_sin", "doy_cos", "is_weekend"]
TOP_DOM   = ["queue_pressure_forecast", "network_pressure_forecast",
             "workload_intensity", "shock_risk", "event_load_forecast"]
TOP3_ROLL = TOP_DOM[:3]
LAGS_DOM, LEADS_DOM, ROLL_WINS = [1, 24], [1, 24], [6, 24]

# MLP config — IDENTICAL to 7e
MLP_HIDDEN          = [256, 256, 128]
MLP_DROPOUT         = 0.2
MLP_EMBED_DIM       = 16
MLP_BATCH           = 1024
MLP_LR              = 1e-3
MLP_WEIGHT_DECAY    = 1e-4
MLP_EPOCHS          = 30
MLP_EARLY_STOP      = 5

# Gate config — FROZEN from 7e
GATE_L1_LAMBDA      = 0.01

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"DATA_DIR    = {DATA_DIR}")
print(f"OUTPUT_DIR  = {OUTPUT_DIR}")
print(f"Seeds       : {SEEDS}")
print(f"Gate L1 λ   : {GATE_L1_LAMBDA} (frozen from 7e)")

DATA_DIR    = /kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data
OUTPUT_DIR  = /kaggle/working
Seeds       : [42, 101, 202, 303, 404]
Gate L1 λ   : 0.01 (frozen from 7e)


---
## §1. Setup & data loading

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import RobustScaler

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

def wape(y_true, y_pred, eps=1e-8):
    return np.sum(np.abs(np.asarray(y_true) - np.asarray(y_pred))) / \
           (np.sum(np.abs(y_true)) + eps) * 100

train_df  = pd.read_csv(TRAIN_PATH,     parse_dates=["timestamp"])
val_input = pd.read_csv(VAL_INPUT_PATH, parse_dates=["timestamp"])
val_index = pd.read_csv(VAL_INDEX_PATH, parse_dates=["timestamp"])

sid_to_code = {sid: i for i, sid in enumerate(sorted(train_df["series_id"].unique()))}
N_SERIES = len(sid_to_code)

print(f"train_df  : {train_df.shape}")
print(f"val_input : {val_input.shape}")
print(f"val_index : {val_index.shape}")
print(f"# series  : {N_SERIES}")

PyTorch device: cuda
  GPU: Tesla T4
train_df  : (414720, 25)
val_input : (32256, 24)
val_index : (32256, 2)
# series  : 96


---
## §2. Feature engineering on the combined train+val frame

Same pipeline as 7e, applied to `pd.concat([train, val_stub])` so lag/lead/rolling features at the start of the val window have proper context from the tail of train (the 7c trick). Then split back into `train_part` (where target is real) and `val_part` (where target is NaN, ready for inference).

In [3]:
def impute_domain_features(df, cols):
    df = df.sort_values(["series_id", "timestamp"]).copy()
    df[cols] = df.groupby("series_id", group_keys=False)[cols].transform(
        lambda x: x.ffill().bfill()
    )
    return df

def add_time_features(df, time_col="timestamp"):
    df = df.copy()
    t = pd.to_datetime(df[time_col])
    df["hour_sin"]   = np.sin(2 * np.pi * t.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * t.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * t.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * t.dt.dayofweek / 7)
    df["doy_sin"]    = np.sin(2 * np.pi * t.dt.dayofyear / 365)
    df["doy_cos"]    = np.cos(2 * np.pi * t.dt.dayofyear / 365)
    df["is_weekend"] = (t.dt.dayofweek >= 5).astype(float)
    return df

def add_temporal_features(df, lag_cols, roll_cols, lags, leads, windows):
    df = df.copy().sort_values(["series_id", "timestamp"])
    new = []
    for col in lag_cols:
        g = df.groupby("series_id", sort=False)[col]
        for L in lags:
            c = f"{col}_lag{L}";  df[c] = g.shift(L);  new.append(c)
        for L in leads:
            c = f"{col}_lead{L}"; df[c] = g.shift(-L); new.append(c)
    for col in roll_cols:
        g = df.groupby("series_id", sort=False)[col]
        for W in windows:
            c_m = f"{col}_roll{W}m"
            df[c_m] = g.shift(1).transform(lambda x: x.rolling(W, min_periods=1).mean())
            new.append(c_m)
            c_s = f"{col}_roll{W}s"
            df[c_s] = g.shift(1).transform(lambda x: x.rolling(W, min_periods=1).std().fillna(0))
            new.append(c_s)
    return df, new

val_stub = val_input.copy()
val_stub["target"] = np.nan
combined = pd.concat([train_df, val_stub], ignore_index=True)
combined = add_time_features(impute_domain_features(combined, DOMAIN_FEAT_COLS))
combined["sid_code"] = combined["series_id"].map(sid_to_code).astype("int32")

sm_full = train_df.groupby("series_id")["target"].mean().rename("series_mean").reset_index()
combined = combined.merge(sm_full, on="series_id")

combined, TEMPORAL_FEATS = add_temporal_features(
    combined, lag_cols=TOP_DOM, roll_cols=TOP3_ROLL,
    lags=LAGS_DOM, leads=LEADS_DOM, windows=ROLL_WINS,
)

NUMERIC_FEATS = DOMAIN_FEAT_COLS + TIME_COLS + ["series_mean"] + TEMPORAL_FEATS

# Gate-index map: which covariate (0..10) each feature derives from, -1 if none.
def build_gate_idx_map(numeric_feats, covariate_cols):
    out = []
    for f in numeric_feats:
        if f in covariate_cols:
            out.append(covariate_cols.index(f))
        else:
            matched = -1
            for ci, c in enumerate(covariate_cols):
                if f.startswith(c + "_"):
                    matched = ci
                    break
            out.append(matched)
    return out

GATE_IDX_MAP = build_gate_idx_map(NUMERIC_FEATS, DOMAIN_FEAT_COLS)

train_part = combined[combined["target"].notna()].reset_index(drop=True)
val_part   = combined[combined["target"].isna()].reset_index(drop=True)

print(f"Numeric features: {len(NUMERIC_FEATS)}")
print(f"Train rows: {len(train_part):,}")
print(f"Val rows:   {len(val_part):,}")
assert len(train_part) == len(train_df), f"train row mismatch: {len(train_part)} vs {len(train_df)}"
assert len(val_part)   == len(val_input), f"val row mismatch: {len(val_part)} vs {len(val_input)}"

Numeric features: 51
Train rows: 414,720
Val rows:   32,256


---
## §3. Model + training utilities — verbatim 7e §5

Same `CovariateGate`, `GatedSeriesMLP`, `prepare_fold_tensors`, `train_gated_mlp_one_seed`. Training loop adds `λ · Σ sigmoid(gates_logits)` to the MAE loss.

In [4]:
class CovariateGate(nn.Module):
    def __init__(self, n_covariates, gate_idx_map):
        super().__init__()
        self.gates_logits = nn.Parameter(torch.zeros(n_covariates))
        self.register_buffer("idx_map", torch.tensor(gate_idx_map, dtype=torch.long))
    def forward(self, x):
        gates_sig = torch.sigmoid(self.gates_logits)
        idx_clamped = self.idx_map.clamp(min=0)
        per_feat = gates_sig[idx_clamped]
        per_feat = torch.where(self.idx_map < 0,
                               torch.ones_like(per_feat),
                               per_feat)
        return x * per_feat.unsqueeze(0)
    def values(self):
        with torch.no_grad():
            return torch.sigmoid(self.gates_logits).cpu().numpy()


class GatedSeriesMLP(nn.Module):
    def __init__(self, n_numeric, n_series, embed_dim, hidden_dims, dropout,
                 n_covariates, gate_idx_map):
        super().__init__()
        self.gate = CovariateGate(n_covariates, gate_idx_map)
        self.series_emb = nn.Embedding(n_series, embed_dim)
        in_dim = n_numeric + embed_dim
        layers = []
        for h in hidden_dims:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h),
                       nn.ReLU(inplace=True), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x_num, sid):
        gated = self.gate(x_num)
        e = self.series_emb(sid)
        return self.net(torch.cat([gated, e], dim=1)).squeeze(-1)


def prepare_tensors(tr_df, te_df, numeric_feats, sid_col="sid_code", target_col="target"):
    feat_scaler = RobustScaler().fit(tr_df[numeric_feats].astype(float))
    tr_X = np.nan_to_num(feat_scaler.transform(tr_df[numeric_feats].astype(float)), nan=0.0)
    te_X = np.nan_to_num(feat_scaler.transform(te_df[numeric_feats].astype(float)), nan=0.0)

    stats = tr_df.groupby(sid_col)[target_col].agg(["mean", "std"]).reset_index()
    stats["std"] = stats["std"].fillna(1.0).clip(lower=1e-3)
    mu = stats.set_index(sid_col)["mean"]
    sd = stats.set_index(sid_col)["std"]
    tr_mu = tr_df[sid_col].map(mu).to_numpy()
    tr_sd = tr_df[sid_col].map(sd).to_numpy()
    te_mu = te_df[sid_col].map(mu).to_numpy()
    te_sd = te_df[sid_col].map(sd).to_numpy()
    tr_y_norm = (tr_df[target_col].to_numpy(float) - tr_mu) / tr_sd

    return dict(
        tr_X=torch.tensor(tr_X, dtype=torch.float32),
        te_X=torch.tensor(te_X, dtype=torch.float32),
        tr_sid=torch.tensor(tr_df[sid_col].to_numpy(np.int64)),
        te_sid=torch.tensor(te_df[sid_col].to_numpy(np.int64)),
        tr_y_norm=torch.tensor(tr_y_norm, dtype=torch.float32),
        te_mu=te_mu, te_sd=te_sd,
    )


def train_gated_mlp_one_seed(tensors, n_numeric, seed, l1_lambda):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = GatedSeriesMLP(
        n_numeric=n_numeric, n_series=N_SERIES,
        embed_dim=MLP_EMBED_DIM, hidden_dims=MLP_HIDDEN, dropout=MLP_DROPOUT,
        n_covariates=len(DOMAIN_FEAT_COLS), gate_idx_map=GATE_IDX_MAP,
    ).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=MLP_LR, weight_decay=MLP_WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MLP_EPOCHS)
    loss_fn = nn.L1Loss()

    n_tr = tensors["tr_X"].shape[0]
    perm = torch.randperm(n_tr, generator=torch.Generator().manual_seed(seed))
    cut = int(0.9 * n_tr)
    tr_idx, val_idx = perm[:cut], perm[cut:]

    tr_loader = DataLoader(
        TensorDataset(
            tensors["tr_X"][tr_idx], tensors["tr_sid"][tr_idx], tensors["tr_y_norm"][tr_idx]
        ),
        batch_size=MLP_BATCH, shuffle=True, drop_last=False,
    )
    val_X = tensors["tr_X"][val_idx].to(DEVICE)
    val_sid = tensors["tr_sid"][val_idx].to(DEVICE)
    val_y = tensors["tr_y_norm"][val_idx].numpy()

    best_val, patience, best_state, used_epochs = float("inf"), 0, None, 0
    for epoch in range(MLP_EPOCHS):
        model.train()
        for x, sid, y in tr_loader:
            x, sid, y = x.to(DEVICE), sid.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            pred = model(x, sid)
            loss = loss_fn(pred, y) + l1_lambda * torch.sum(torch.sigmoid(model.gate.gates_logits))
            loss.backward()
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(val_X, val_sid).cpu().numpy()
        val_mae = float(np.mean(np.abs(val_pred - val_y)))

        if val_mae < best_val - 1e-5:
            best_val   = val_mae
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience   = 0
            used_epochs = epoch + 1
        else:
            patience += 1
            if patience >= MLP_EARLY_STOP:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        te_X = tensors["te_X"].to(DEVICE)
        te_sid = tensors["te_sid"].to(DEVICE)
        te_pred_norm = model(te_X, te_sid).cpu().numpy()
    te_pred = te_pred_norm * tensors["te_sd"] + tensors["te_mu"]
    te_pred = np.clip(te_pred, 0, None)
    return te_pred, model.gate.values(), best_val, used_epochs

print("Model + training utilities ready.")

Model + training utilities ready.


---
## §4. Refit × 5 seeds on full train → predict val

Each seed re-learns its own gates. We track gates per seed for the components CSV — they should land near the 7e CV values (top-3: queue_pressure, event_load, network_pressure; bottom-2: service_irregularity_risk, unit_reliability). If a seed's gates look wildly different, that's a signal to investigate.

In [5]:
import time

print(f"Preparing tensors (train: {len(train_part):,} rows, val: {len(val_part):,} rows)...")
tensors = prepare_tensors(train_part, val_part, NUMERIC_FEATS)

print(f"\nFitting GatedSeriesMLP × {len(SEEDS)} seeds on full train (L1 λ = {GATE_L1_LAMBDA})...\n")
t0 = time.time()
seed_preds, seed_gates = [], []
for s in SEEDS:
    seed_t = time.time()
    p, gates, val_loss, used_epochs = train_gated_mlp_one_seed(
        tensors, n_numeric=len(NUMERIC_FEATS),
        seed=s, l1_lambda=GATE_L1_LAMBDA,
    )
    seed_preds.append(p)
    seed_gates.append(gates)
    print(f"  seed {s:>4}: val_L1 = {val_loss:.4f}, epochs = {used_epochs}, "
          f"mean_pred = {p.mean():.3f}, time = {time.time() - seed_t:.1f}s")
    # Show top-3 and bottom-3 gates for this seed
    order = np.argsort(-gates)
    top3 = ", ".join([f"{DOMAIN_FEAT_COLS[i]}={gates[i]:.3f}" for i in order[:3]])
    bot3 = ", ".join([f"{DOMAIN_FEAT_COLS[i]}={gates[i]:.3f}" for i in order[-3:]])
    print(f"           top-3: {top3}")
    print(f"           bot-3: {bot3}")

pred_mlp = np.mean(np.stack(seed_preds, axis=0), axis=0)
pred_mlp = np.clip(pred_mlp, 0.0, None)
gate_matrix = np.stack(seed_gates, axis=0)   # (5, 11)

print(f"\nEnsemble done in {time.time() - t0:.1f}s.")
print(f"  pred range: [{pred_mlp.min():.3f}, {pred_mlp.max():.3f}], mean {pred_mlp.mean():.3f}")

# Summary of gates across all 5 seeds (the final, shippable feature-importance answer)
gate_mean = gate_matrix.mean(axis=0)
gate_std  = gate_matrix.std(axis=0)
order = np.argsort(-gate_mean)
print("\n=== Final learned gates (averaged over 5 production seeds) ===")
print(f"  {'covariate':<40}  {'mean':>6}  {'std':>6}")
print("  " + "-" * 60)
for i in order:
    print(f"  {DOMAIN_FEAT_COLS[i]:<40}  {gate_mean[i]:>6.3f}  {gate_std[i]:>6.3f}")

Preparing tensors (train: 414,720 rows, val: 32,256 rows)...

Fitting GatedSeriesMLP × 5 seeds on full train (L1 λ = 0.01)...

  seed   42: val_L1 = 0.2737, epochs = 30, mean_pred = 10.669, time = 158.9s
           top-3: queue_pressure_forecast=0.205, event_load_forecast=0.158, network_pressure_forecast=0.134
           bot-3: workload_intensity=0.045, service_irregularity_risk_forecast=0.022, unit_reliability_forecast=0.018
  seed  101: val_L1 = 0.2730, epochs = 26, mean_pred = 10.675, time = 154.1s
           top-3: queue_pressure_forecast=0.204, event_load_forecast=0.164, network_pressure_forecast=0.134
           bot-3: workload_intensity=0.045, service_irregularity_risk_forecast=0.022, unit_reliability_forecast=0.019
  seed  202: val_L1 = 0.2741, epochs = 30, mean_pred = 10.659, time = 150.9s
           top-3: queue_pressure_forecast=0.204, event_load_forecast=0.160, network_pressure_forecast=0.132
           bot-3: workload_intensity=0.043, service_irregularity_risk_forecast=0.0

---
## §5. Sanity-check, align with forecast index, write submission

In [6]:
# Distribution sanity
print("Prediction distribution (mean, std, p50, p95):")
print(f"  pred_mlp: mean = {pred_mlp.mean():.3f}, std = {pred_mlp.std():.3f}, "
      f"p50 = {np.median(pred_mlp):.3f}, p95 = {np.percentile(pred_mlp, 95):.3f}")

# Per-series drift vs train per-series mean
train_per_sid = train_df.groupby("series_id")["target"].mean()
val_per_sid   = pd.Series(pred_mlp, index=val_part["series_id"].values).groupby(level=0).mean()
rel_drift = ((val_per_sid - train_per_sid) / train_per_sid.replace(0, np.nan)).abs()
print(f"\nPer-series |val_pred_mean − train_target_mean| / train_target_mean:")
print(f"  median = {rel_drift.median():.3f}, p90 = {rel_drift.quantile(0.9):.3f}, max = {rel_drift.max():.3f}")
if rel_drift.median() > 0.5:
    print("  ⚠️  Median per-series drift > 50% — investigate before submitting.")

# Build submission
val_rows = pd.DataFrame({
    "series_id": val_part["series_id"].values,
    "timestamp": val_part["timestamp"].values,
    "prediction": pred_mlp,
})
submission = val_index.merge(val_rows, on=["series_id", "timestamp"], how="left")
missing = submission["prediction"].isna().sum()
if missing > 0:
    print(f"\n⚠️  {missing} rows have no prediction after merge — filling with series_mean.")
    sm_map = sm_full.set_index("series_id")["series_mean"]
    submission["prediction"] = submission["prediction"].fillna(submission["series_id"].map(sm_map))
submission["prediction"] = submission["prediction"].clip(lower=0.0)

assert submission["prediction"].isna().sum() == 0, "prediction column still has NaNs"
assert len(submission) == len(val_index), "submission row count diverged from val_index"

# Write outputs
sub_path  = OUTPUT_DIR / "validation_predictions_7f.csv"
comp_path = OUTPUT_DIR / "val_components_7f.csv"
gate_path = OUTPUT_DIR / "gates_7f.csv"

submission[["series_id", "timestamp", "prediction"]].to_csv(sub_path, index=False)

# Per-seed components: predictions per seed + ensemble + per-seed gate values
comp_df = pd.DataFrame({
    "series_id": val_part["series_id"].values,
    "timestamp": val_part["timestamp"].values,
    "pred_ensemble": pred_mlp,
})
for i, s in enumerate(SEEDS):
    comp_df[f"pred_seed_{s}"] = seed_preds[i]
comp_df.to_csv(comp_path, index=False)

# Per-seed gate values + summary row
gate_rows = []
for i, s in enumerate(SEEDS):
    for j, name in enumerate(DOMAIN_FEAT_COLS):
        gate_rows.append({"seed": s, "feature": name, "gate": float(seed_gates[i][j])})
gate_df_long = pd.DataFrame(gate_rows)
gate_df_long.to_csv(gate_path, index=False)

print(f"\n✅ Wrote {len(submission):,} predictions → {sub_path}")
print(f"✅ Wrote per-seed components → {comp_path}")
print(f"✅ Wrote per-seed gates → {gate_path}")
print("\nFirst 5 submission rows:")
print(submission[["series_id", "timestamp", "prediction"]].head())

Prediction distribution (mean, std, p50, p95):
  pred_mlp: mean = 10.659, std = 5.233, p50 = 9.837, p95 = 20.537

Per-series |val_pred_mean − train_target_mean| / train_target_mean:
  median = 0.059, p90 = 0.257, max = 0.466

✅ Wrote 32,256 predictions → /kaggle/working/validation_predictions_7f.csv
✅ Wrote per-seed components → /kaggle/working/val_components_7f.csv
✅ Wrote per-seed gates → /kaggle/working/gates_7f.csv

First 5 submission rows:
  series_id           timestamp  prediction
0  unit_000 2023-06-30 00:00:00    4.393902
1  unit_000 2023-06-30 01:00:00    4.901173
2  unit_000 2023-06-30 02:00:00    5.503122
3  unit_000 2023-06-30 03:00:00    4.889140
4  unit_000 2023-06-30 04:00:00    5.404753


---
## §6. Interpreting the leaderboard result

- **LB < 14.7** : the gated MLP slightly outperformed the ensemble (LB 14.68). Pure DL beat the boosted blend on this dataset. Strongest possible writeup.
- **LB 14.7 – 14.8** : on-trend with CV (14.59) given the typical 0.1–0.2 CV→LB gap. Pure-DL submission is competitive with the ensemble. Solid writeup.
- **LB 14.8 – 15.0** : larger-than-expected CV→LB gap, but the model still beat the LGBM-only baseline (LB 14.92 from exp-5). Acceptable pure-DL artifact.
- **LB > 15.0** : something diverged between CV and refit. Most likely: the 5-seed refit on full train under-trained (early-stop fired too early because the inner-val 10% holdout is harder than the CV val slice). The `val_components_7f.csv` per-seed predictions let you diagnose without re-running.

Whichever number comes back, the gate values in `gates_7f.csv` are the feature-selection deliverable for the professor — independent of the WAPE outcome.